In [ ]:
# Fix accelerate version
!pip uninstall -y accelerate
!pip install -q accelerate==0.28.0

In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

In [1]:
import numpy
print(numpy.__version__)

1.26.4


In [2]:
import accelerate
print(accelerate.__version__)

0.28.0


In [ ]:
!pip uninstall transformers -y
!pip uninstall peft -y

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2
!pip install transformers==4.38.2
!pip install peft==0.10.0

In [3]:
import transformers
print(transformers.__version__)

4.38.2


In [4]:
import torch
print(torch.__version__)

2.2.2+cu121


In [5]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/itslakhvir/human-stress-detection/Stress.csv")

df = df[["text", "label"]].dropna()

print(df.head())

                                                text  label
0  He said he had not felt that way before, sugge...      1
1  Hey there r/assistance, Not sure if this is th...      0
2  My mom then hit me with the newspaper and it s...      1
3  until i met my new boyfriend, he is amazing, h...      1
4  October is Domestic Violence Awareness Month a...      1


In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
import torch

class StressDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = StressDataset(train_encodings, train_labels)
val_dataset = StressDataset(val_encodings, val_labels)

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

2026-04-14 19:42:15.581498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776195735.602133    1516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776195735.608472    1516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776195735.625111    1516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776195735.625134    1516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776195735.625137    1516 computation_placer.cc:177] computation placer alr

In [13]:
import numpy
trainer.train()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss
1,No log,0.406059
2,No log,0.445976
3,No log,0.681855
4,0.282400,0.861356
5,0.282400,0.897813


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead u

TrainOutput(global_step=710, training_loss=0.21277862736876582, metrics={'train_runtime': 288.2248, 'train_samples_per_second': 39.379, 'train_steps_per_second': 2.463, 'total_flos': 746577619584000.0, 'train_loss': 0.21277862736876582, 'epoch': 5.0})

In [14]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


{'eval_loss': 0.8978134989738464,
 'eval_runtime': 3.3489,
 'eval_samples_per_second': 169.608,
 'eval_steps_per_second': 10.75,
 'epoch': 5.0}

In [17]:
preds = trainer.predict(val_dataset)

y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.79      0.80       270
           1       0.81      0.83      0.82       298

    accuracy                           0.81       568
   macro avg       0.81      0.81      0.81       568
weighted avg       0.81      0.81      0.81       568



In [15]:
model.save_pretrained("bert_stress_model")
tokenizer.save_pretrained("bert_stress_model")

('bert_stress_model/tokenizer_config.json',
 'bert_stress_model/special_tokens_map.json',
 'bert_stress_model/vocab.txt',
 'bert_stress_model/added_tokens.json',
 'bert_stress_model/tokenizer.json')

In [16]:
import shutil

shutil.make_archive("bert_stress_model", "zip", "bert_stress_model")

'/kaggle/working/bert_stress_model.zip'